# EXP-002 Task Profiler and Replay Analysis

Analysis notebook for all 25 public ARC-AGI-3 tasks. It saves metadata, action spaces, initial frame objects, one-step probe effects, and task-family hints. Do not use as the scoring submission notebook.


In [ ]:
!python -m pip install -q --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv pyarrow


In [ ]:
import os, json, zlib
from pathlib import Path
from collections import deque, defaultdict
import numpy as np, pandas as pd, dotenv, arc_agi
from arcengine import GameAction, GameState

ART=Path('/kaggle/working/exp002_profiler_artifacts'); ART.mkdir(exist_ok=True)
WORK=Path('/kaggle/working')
envdir='/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/'
(WORK/'.env').write_text(f'''SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE=offline\nENVIRONMENTS_DIR={envdir}\nRECORDINGS_DIR=/kaggle/working/server_recording\n''')
dotenv.load_dotenv(WORK/'.env', override=True)
MEANING={'ACTION1':'up','ACTION2':'down','ACTION3':'left','ACTION4':'right','ACTION5':'spacebar','ACTION6':'click_xy','ACTION7':'undo'}
def fh(f): return None if f is None else int(zlib.adler32(np.asarray(f,dtype=np.uint8).tobytes()) & 0xffffffff)
def last_frame(obs): return None if obs is None or not getattr(obs,'frame',None) else np.asarray(obs.frame[-1])
def reset(env):
    obs=getattr(env,'_last_response',None)
    if obs is None or obs.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
        try: obs=env.step(GameAction.RESET,{})
        except Exception: pass
    return obs
def diff(a,b):
    if a is None or b is None or a.shape!=b.shape: return {'frame_changed':None,'diff_pixels':None,'diff_cx':None,'diff_cy':None}
    d=a!=b; ys,xs=np.where(d)
    return {'frame_changed':bool(len(xs)>0),'diff_pixels':int(len(xs)),'diff_cx':float(xs.mean()) if len(xs) else None,'diff_cy':float(ys.mean()) if len(ys) else None}
def hist(frame,gid):
    vals,cnts=np.unique(frame,return_counts=True); total=frame.size
    return [{'game_id':gid,'color':int(v),'count':int(c),'share':float(c/total)} for v,c in zip(vals,cnts)]
def comps(frame,gid,min_area=2,max_area=750):
    arr=np.asarray(frame); h,w=arr.shape[:2]; vals,cnts=np.unique(arr,return_counts=True); cc=dict(zip(vals.tolist(),cnts.tolist()))
    ignore=set([c for c,n in sorted(cc.items(), key=lambda kv:kv[1], reverse=True)[:2]]) | set(c for c,n in cc.items() if n>0.30*h*w)
    seen=np.zeros((h,w),bool); rows=[]; cid=0
    for y in range(h):
      for x in range(w):
        if seen[y,x]: continue
        col=int(arr[y,x])
        if col in ignore: seen[y,x]=1; continue
        q=deque([(x,y)]); seen[y,x]=1; pts=[]
        while q:
          cx,cy=q.popleft(); pts.append((cx,cy))
          for nx,ny in ((cx+1,cy),(cx-1,cy),(cx,cy+1),(cx,cy-1)):
            if 0<=nx<w and 0<=ny<h and not seen[ny,nx] and int(arr[ny,nx])==col:
              seen[ny,nx]=1; q.append((nx,ny))
        if min_area<=len(pts)<=max_area:
          xs=[p[0] for p in pts]; ys=[p[1] for p in pts]; cid+=1
          rows.append({'game_id':gid,'component_id':cid,'color':col,'area':len(pts),'x0':min(xs),'x1':max(xs),'y0':min(ys),'y1':max(ys),'cx':sum(xs)/len(xs),'cy':sum(ys)/len(ys),'width':max(xs)-min(xs)+1,'height':max(ys)-min(ys)+1,'touches_edge':min(xs)==0 or min(ys)==0 or max(xs)==w-1 or max(ys)==h-1})
    return rows
def candidates(rows, n=6):
    scored=[]
    for r in rows:
      if r['cy']>61 or r['area']>200 or r['width']>25 or r['height']>25: continue
      s=1/(1+abs(r['area']-12)) + (0.2 if not r['touches_edge'] else -0.2)
      scored.append((s,r))
    return [r for s,r in sorted(scored,key=lambda x:x[0],reverse=True)[:n]]
def probe(gid, action_name, data=None):
    arc=arc_agi.Arcade(); env=arc.make(gid); o0=reset(env); f0=last_frame(o0); lvl0=int(o0.levels_completed) if o0 else None
    err=None; o1=None
    try: o1=env.step(getattr(GameAction,action_name), data=data or {})
    except Exception as e: err=repr(e)
    f1=last_frame(o1); row={'game_id':gid,'action':action_name,'meaning':MEANING.get(action_name,''),'data':json.dumps(data or {},sort_keys=True),'state_before':o0.state.name if o0 else None,'state_after':o1.state.name if o1 else None,'levels_before':lvl0,'levels_after':int(o1.levels_completed) if o1 else None,'level_delta':(int(o1.levels_completed)-lvl0) if o1 and lvl0 is not None else None,'error':err,'hash_before':fh(f0),'hash_after':fh(f1)}
    row.update(diff(f0,f1)); return row
print('ready', ART)


In [ ]:
arc=arc_agi.Arcade(); games=list(arc.get_environments()); print('games',len(games))
meta=[]; actions=[]; hrows=[]; crows=[]; candrows=[]
for i,info in enumerate(games,1):
    gid=info.game_id; env=arc.make(gid); obs=reset(env); frame=last_frame(obs); acts=list(getattr(env,'action_space',[]))
    tags=getattr(info,'tags',[]); tags=','.join(tags) if isinstance(tags,(list,tuple)) else str(tags)
    meta.append({'game_id':gid,'title':getattr(info,'title',''),'tags':tags,'initial_state':obs.state.name if obs else None,'initial_levels_completed':int(obs.levels_completed) if obs else None,'n_actions':len(acts)})
    for a in acts: actions.append({'game_id':gid,'action':a.name,'meaning':MEANING.get(a.name,''),'is_complex':bool(a.is_complex())})
    if frame is not None:
        hrows += hist(frame,gid); cr=comps(frame,gid); crows += cr
        for rank,r in enumerate(candidates(cr),1): rr=dict(r); rr['candidate_rank']=rank; candrows.append(rr)
    print(f'[{i:02d}/{len(games):02d}] {gid} tags={tags} actions={[a.name for a in acts]}')
pd.DataFrame(meta).to_csv(ART/'game_metadata.csv',index=False)
pd.DataFrame(actions).to_csv(ART/'action_space_by_game.csv',index=False)
pd.DataFrame(hrows).to_csv(ART/'initial_frame_color_histograms.csv',index=False)
pd.DataFrame(crows).to_csv(ART/'initial_connected_components.csv',index=False)
pd.DataFrame(candrows).to_csv(ART/'initial_target_candidates.csv',index=False)
pd.DataFrame(meta).head()


In [ ]:
probe_rows=[]
for i,info in enumerate(games,1):
    gid=info.game_id; avail=[r['action'] for r in actions if r['game_id']==gid]
    for an in [a for a in avail if a!='RESET' and a!='ACTION6']:
        probe_rows.append(probe(gid,an,{}))
    if 'ACTION6' in avail:
        for cand in [r for r in candrows if r['game_id']==gid][:6]:
            data={'x':int(round(cand['cx'])),'y':int(round(cand['cy']))}; row=probe(gid,'ACTION6',data)
            row.update({'clicked_component_id':cand['component_id'],'clicked_color':cand['color'],'clicked_area':cand['area'],'clicked_cx':cand['cx'],'clicked_cy':cand['cy']}); probe_rows.append(row)
    print(f'[{i:02d}/{len(games):02d}] probed {gid}')
probe_df=pd.DataFrame(probe_rows); probe_df.to_csv(ART/'probe_action_effects.csv',index=False)
probe_df.head(20)


In [ ]:
probe_df=pd.read_csv(ART/'probe_action_effects.csv'); meta_df=pd.read_csv(ART/'game_metadata.csv'); actions_df=pd.read_csv(ART/'action_space_by_game.csv'); comp_df=pd.read_csv(ART/'initial_connected_components.csv')
families=[]
for gid in meta_df.game_id.tolist():
    gp=probe_df[probe_df.game_id==gid]; ga=actions_df[actions_df.game_id==gid]; gc=comp_df[comp_df.game_id==gid]
    moves=['ACTION1','ACTION2','ACTION3','ACTION4']; has_move=bool(ga.action.isin(moves).any()); has_click=bool((ga.action=='ACTION6').any())
    fam='keyboard_click' if has_move and has_click else ('click' if has_click else ('keyboard' if has_move else 'unknown'))
    mchg=int(gp[(gp.action.isin(moves)) & (gp.frame_changed==True)].shape[0]); cchg=int(gp[(gp.action=='ACTION6') & (gp.frame_changed==True)].shape[0]); schg=int(gp[(gp.action=='ACTION5') & (gp.frame_changed==True)].shape[0]); prog=int(gp[gp.level_delta.fillna(0)>0].shape[0])
    hints=[]
    if mchg: hints.append('movement_responsive')
    if cchg: hints.append('click_responsive')
    if schg: hints.append('space_responsive')
    if prog: hints.append('one_step_progress')
    if int(gc[(gc.area<=30)&(gc.cy<=61)].shape[0])>=3: hints.append('many_small_objects')
    families.append({'game_id':gid,'coarse_family':fam,'hints':','.join(hints),'has_movement':has_move,'has_click':has_click,'movement_changed_count':mchg,'click_changed_count':cchg,'space_changed_count':schg,'one_step_level_delta_count':prog,'n_components':int(gc.shape[0]),'small_components':int(gc[(gc.area<=30)&(gc.cy<=61)].shape[0])})
fam_df=pd.DataFrame(families); fam_df.to_csv(ART/'task_family_summary.csv',index=False)
summary={'exp_id':'EXP-002-PROFILER','n_games':len(meta_df),'n_probe_rows':len(probe_df),'coarse_family_counts':fam_df.coarse_family.value_counts().to_dict(),'movement_responsive_games':fam_df[fam_df.movement_changed_count>0].game_id.tolist(),'click_responsive_games':fam_df[fam_df.click_changed_count>0].game_id.tolist(),'one_step_progress_games':fam_df[fam_df.one_step_level_delta_count>0].game_id.tolist()}
(ART/'profiler_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2)); fam_df


In [ ]:
manifest=sorted(str(p) for p in ART.glob('*')); pd.DataFrame({'artifact':manifest}).to_csv(ART/'artifact_manifest.csv',index=False)
print('Artifacts:'); print('\n'.join(manifest))
sp=Path('/kaggle/working/submission.parquet')
if not sp.exists(): pd.DataFrame([['1_0','1',True,1]],columns=['row_id','game_id','end_of_game','score']).to_parquet(sp,index=False)
print('Profiler complete. Do not submit this notebook as scoring notebook.')
